[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Connecting and Executing &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: a server, the `guide` database and five thousand events. Run
it first. The tasks only read, apart from task 5, so they can be run in any order.


In [1]:
import getpass
import os
import subprocess
import sys
import time
from dataclasses import dataclass
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg.rows import class_row, dict_row, namedtuple_row, scalar_row

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


**1.** A count per kind, as dictionaries.


In [2]:
with psycopg.connect("dbname=guide", row_factory=dict_row) as conn:
    rows = conn.execute("SELECT kind, count(*) AS total FROM events GROUP BY kind ORDER BY kind").fetchall()

for row in rows:
    print(f"  {row['kind']:<9} {row['total']}")
print(rows[0])


  click     1666
  purchase  1667
  view      1667
{'kind': 'click', 'total': 1666}


`row_factory` on the connection means every cursor made from it gives dictionaries, so nothing
further down has to remember. The `AS total` matters: without it the column would be called `count`,
and that is the key you would have to use.


**2.** Two cursors, one connection.


In [3]:
conn = psycopg.connect("dbname=guide")

with conn.cursor() as first:
    print("kinds: ", first.execute("SELECT count(DISTINCT kind) FROM events").fetchone())
with conn.cursor() as second:
    print("events:", second.execute("SELECT count(*) FROM events").fetchone())

print("both cursors closed:", first.closed and second.closed)
print("the connection open:", not conn.closed)
conn.close()


kinds:  (3,)
events: (5000,)
both cursors closed: True
the connection open: True


Cursors are cheap and short-lived, and the connection outlives them. That is the shape most code
wants: one connection held for a while, cursors made and dropped inside it.


**3.** The same four rows, taken three ways.


In [4]:
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    cur.execute("SELECT id, kind FROM events ORDER BY id LIMIT 4")
    one = cur.fetchone()
    two = cur.fetchmany(2)
    rest = cur.fetchall()

print("fetchone():  ", one, "  <- 1 row, the first")
print("fetchmany(2):", two, "<- the next 2")
print("fetchall():  ", rest, "  <- the 1 that was left, not all 4")


fetchone():   (1, 'view')   <- 1 row, the first
fetchmany(2): [(2, 'purchase'), (3, 'click')] <- the next 2
fetchall():   [(4, 'view')]   <- the 1 that was left, not all 4


Every call takes rows off the front of the same result, so they add up to the four the query asked
for. `fetchall` is only "all" when nothing has been read yet.


**4.** Rows as a class of your own.


In [5]:
@dataclass
class Happening:
    id: int
    kind: str
    payload: dict


with psycopg.connect("dbname=guide") as conn:
    with conn.cursor(row_factory=class_row(Happening)) as cur:
        rows = cur.execute("SELECT id, kind, payload FROM events ORDER BY id LIMIT 3").fetchall()

for row in rows:
    print(" ", row)
print("real instances:", all(isinstance(row, Happening) for row in rows))


  Happening(id=1, kind='view', payload={'n': 1, 'size': 2})
  Happening(id=2, kind='purchase', payload={'n': 2, 'size': 3})
  Happening(id=3, kind='click', payload={'n': 3, 'size': 4})
real instances: True


The column names and the field names have to line up, which is the whole contract of `class_row`.
Select a column the class has no field for and it raises rather than quietly dropping it.


**5.** A write that does not survive.


In [6]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("DROP TABLE IF EXISTS attempts")
    conn.execute("CREATE TABLE attempts (id int, note text)")


def count():
    with psycopg.connect("dbname=guide") as conn:
        return conn.execute("SELECT count(*) FROM attempts").fetchone()[0]


try:
    with psycopg.connect("dbname=guide") as conn:
        conn.execute("INSERT INTO attempts VALUES (1, 'written')")
        print("inside the block, the row is there:",
              conn.execute("SELECT count(*) FROM attempts").fetchone()[0])
        raise RuntimeError("and then something failed")
except RuntimeError as error:
    print("the block was left by:", error)

print("afterwards:", count(), "rows")


inside the block, the row is there: 1
the block was left by: and then something failed
afterwards: 0 rows


The row was visible inside the block because the transaction that wrote it could see its own work.
Nobody else ever could, and the exception on the way out rolled it back.


**6.** Three shapes from asyncpg.


In [7]:
conn = await asyncpg.connect(database="guide")

rows = await conn.fetch("SELECT id, kind FROM events ORDER BY id LIMIT 2")
row = await conn.fetchrow("SELECT id, kind FROM events ORDER BY id LIMIT 1")
value = await conn.fetchval("SELECT count(*) FROM events")

print("fetch    ->", type(rows).__name__, "of", type(rows[0]).__name__, "|", [dict(r) for r in rows])
print("fetchrow ->", type(row).__name__, "|", dict(row))
print("fetchval ->", type(value).__name__, "|", value)
await conn.close()


fetch    -> list of Record | [{'id': 1, 'kind': 'view'}, {'id': 2, 'kind': 'purchase'}]
fetchrow -> Record | {'id': 1, 'kind': 'view'}
fetchval -> int | 5000


Three methods rather than one method and a row factory. `fetch` gives a list of `Record`, `fetchrow`
gives one `Record` or `None`, and `fetchval` gives the first column of the first row, which is
psycopg's `scalar_row` without the ceremony.


---

&#8592; **Back to:** [Connecting and Executing](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/02-connecting-and-executing.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
